# 9. 어노테이터 실 LLM 스모크 (M3 4c)

합성 픽스처 **fx1**(PII 없음)에 실제 `annotate`를 1건 돌려 structured output·모델
(`claude-sonnet-4-5`)·LangSmith 트래킹·V2를 검증한다. 키는 저장소 루트 `.env`
(gitignore됨)에서 로드한다. **코퍼스(실 PII)는 여기서 쓰지 않는다** — 32종 시드는
맨 아래 안내대로 로컬에서만 실행하고 출력/원문을 커밋하지 않는다.

> 커밋 시 실행 출력은 비운다(fx1은 합성이라 무해하나 관례 유지).

In [ ]:
# .env 로드 + src 경로 등록 (저장소 루트를 위로 탐색)
import os, sys
from pathlib import Path

repo_root = None
for p in [Path.cwd(), *Path.cwd().parents]:
    if (p / '.env').exists():
        repo_root = p
        for line in (p / '.env').read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, v = line.split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
        break
assert repo_root, '.env를 찾지 못함 — 저장소 루트에서 실행하세요'
sys.path.insert(0, str(repo_root / 'src'))
print('repo_root =', repo_root)
print('ANTHROPIC_API_KEY 설정:', bool(os.environ.get('ANTHROPIC_API_KEY')))
print('LangSmith 트래킹:', os.environ.get('LANGCHAIN_TRACING_V2'), '| project:', os.environ.get('LANGCHAIN_PROJECT'))

In [ ]:
# fx1 → 결정론 패키지(임시 폴더)
import tempfile, json
from excel_to_skill.cli import _convert_one
from excel_to_skill.meta import _converter_version

workdir = Path(tempfile.mkdtemp(prefix='annotate_live_'))
src = repo_root / 'tests' / 'fixtures' / 'fx1_merge_formula.xlsx'
pkg = _convert_one(src, workdir, force=True, cv=_converter_version())
print('패키지:', pkg)

In [ ]:
# 실 LLM annotate (client=None → anthropic + structured output + LangSmith)
from excel_to_skill.annotator import annotate_package, DEFAULT_MODEL

print('model =', DEFAULT_MODEL)
res = annotate_package(pkg)
print('cached=%s  sheets=%s  excluded=%s' % (res.get('cached'), res['sheets'], res['excluded']))
sem = json.loads((pkg / 'data/semantics.json').read_text(encoding='utf-8'))
print('generator.model =', sem['generator']['model'])
print('review.status   =', sem['review']['status'])
print('workbook_claims =', len(sem['workbook_claims']), '| sheets 주석 =', len(sem['sheets']))

In [ ]:
# verify — V1 스키마 / V2 실재성 / annotation 일관성
from excel_to_skill.verify import verify_package

result = verify_package(pkg)
for c in result.checks:
    mark = 'SKIP' if c.skipped else ('PASS' if c.ok else 'FAIL')
    print(f'  [{mark}] {c.name}: {c.detail}')
print('verify ok =', result.ok)

## 32종 코퍼스 draft 시드 (로컬 전용 — 커밋 금지)

실 PII 코퍼스(`workingpaper_raw/`, gitignore됨)에 대한 시드는 **이 노트북 밖 또는
아래 셀을 로컬에서만** 실행하고, **출력·원문(파일명·셀 내용·SKILL 본문)을 저장소에
남기지 않는다.** 보고는 카운트·상태만.

```bash
# 1) 전수 결정론 변환
uv run excel-to-skill convert workingpaper_raw --all --out ./_seed_out
# 2) 전 패키지 draft 주석(배치) — 캐시 hit 생략·파일별 실패 격리·집계
uv run excel-to-skill annotate ./_seed_out --all
#    stdout=요약 카운트 한 줄(패키지명 없음), per-package·요약은 stderr.
#    exit 1이면 제외/실패 존재.
```